In [1]:
import os
import kagglehub
import shutil
import pandas as pd


path = kagglehub.competition_download('digit-recognizer')

print("Fichiers téléchargés dans :", path)
# Dossier de destination
dest = "data"
#os.makedirs(dest, exist_ok=True)

# Copier tous les fichiers du dossier téléchargé vers /data
for fichier in os.listdir(path):
    src_file = os.path.join(path, fichier)
    dst_file = os.path.join(dest, fichier)
    shutil.copy2(src_file, dst_file)

print("Fichiers copiés dans :", dest)

/opt/python/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 15.3M/15.3M [00:01<00:00, 15.1MB/s]

Extracting files...


Fichiers téléchargés dans : /home/onyxia/.cache/kagglehub/competitions/digit-recognizer
Fichiers copiés dans : data


In [11]:
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from scikeras.wrappers import KerasClassifier
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import GridSearchCV

In [2]:
df_train=pd.read_csv('data/train.csv')
df_test=pd.read_csv('data/test.csv')

In [5]:
df_train.head()

,label,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [12]:
df_test.describe()

,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
count,28000.0,28000.0,28000.0,28000.0,28000.0,28000.0,28000.0,28000.0,28000.0,28000.0,...,28000.000000,28000.000000,28000.000000,28000.000000,28000.000000,28000.0,28000.0,28000.0,28000.0,28000.0
mean,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.164607,0.073214,0.028036,0.011250,0.006536,0.0,0.0,0.0,0.0,0.0
std,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,5.473293,3.616811,1.813602,1.205211,0.807475,0.0,0.0,0.0,0.0,0.0
min,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
25%,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
50%,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
75%,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
max,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,253.000000,254.000000,193.000000,187.000000,119.000000,0.0,0.0,0.0,0.0,0.0


In [8]:
X_train=df_train.drop(columns=['label'])
X_test=df_test
y_train=df_train['label']

In [15]:
def build_model(meta, hidden_units=64):
    n_features_in = meta["n_features_in_"]
    n_classes = meta["n_classes_"]
    model = keras.Sequential([
        keras.Input(shape=(n_features_in,)),
        layers.Dense(hidden_units, activation="relu"),
        layers.Dense(n_classes, activation="softmax"),
    ])
    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


clf = KerasClassifier(
    model=build_model,
    hidden_units=64,
    epochs=10,
    batch_size=30,
    validation_split=0.2,
    verbose=-1,
)



pipeline = Pipeline([
    ("scaler", MinMaxScaler()),
    ("model", clf),
])

param_grid = {
    "model__hidden_units": [32, 64, 128]
}
 
grid = GridSearchCV(pipeline, param_grid, cv=3, scoring="accuracy", n_jobs=-1)
grid.fit(X_train, y_train)
preds = grid.predict(X_test)

I0000 00:00:1784124940.620654  685648 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1784124940.622247  685648 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1784124940.634871  685645 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1784124940.634871  685640 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1784124940.635793  68

Epoch 1/10
Epoch 1/10
Epoch 1/10
Epoch 1/10
Epoch 1/10
Epoch 1/10
Epoch 1/10


E0000 00:00:1784124943.721557  685650 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Epoch 1/10


E0000 00:00:1784124944.418400  685649 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Epoch 1/10
Epoch 2/10
Epoch 2/10
Epoch 2/10
Epoch 2/10
Epoch 2/10
Epoch 2/10
Epoch 2/10
Epoch 2/10
Epoch 2/10
Epoch 3/10
Epoch 3/10
Epoch 3/10
Epoch 3/10
Epoch 3/10
Epoch 3/10
Epoch 3/10
Epoch 3/10
Epoch 3/10
Epoch 4/10
Epoch 4/10
Epoch 4/10
Epoch 4/10
Epoch 4/10
Epoch 5/10
Epoch 5/10
Epoch 4/10
Epoch 4/10
Epoch 4/10
Epoch 5/10
Epoch 4/10
Epoch 5/10
Epoch 5/10
Epoch 6/10
Epoch 6/10
Epoch 5/10
Epoch 6/10
Epoch 5/10
Epoch 5/10
Epoch 5/10
Epoch 6/10
Epoch 7/10
Epoch 6/10
Epoch 7/10
Epoch 6/10
Epoch 7/10
Epoch 7/10
Epoch 8/10
Epoch 6/10
Epoch 8/10
Epoch 6/10
Epoch 7/10
Epoch 6/10
Epoch 7/10
Epoch 8/10
Epoch 9/10
Epoch 8/10
Epoch 9/10
Epoch 8/10
Epoch 7/10
Epoch 7/10
Epoch 9/10
Epoch 8/10
Epoch 7/10
Epoch 10/10
Epoch 10/10
Epoch 9/10
Epoch 9/10
Epoch 8/10
Epoch 10/10
Epoch 9/10
Epoch 8/10
Epoch 8/10
Epoch 10/10
Epoch 10/10
Epoch 10/10
Epoch 9/10
Epoch 9/10
Epoch 9/10
Epoch 10/10
Epoch 10/10
Epoch 10/10
Epoch 1/10
Epoch 2/10
Epoch 3/10
Epoch 4/10
Epoch 5/10
Epoch 6/10
Epoch 7/10
Epoch 8/10
E

In [ ]:
submission = pd.DataFrame({
    "ImageId": np.arange(1, len(preds) + 1),   # ou range(len(preds)) selon le format attendu
    "Label": preds
})
submission.to_csv('soumission_test.csv',index=False)

In [17]:
submission.to_csv('soumission_test.csv',index=False)